## Age Predictor With Synthetic Data

# STEP 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

#STEP 2: Import Required Libraries

In [ ]:
import os
import re
from sklearn.utils import shuffle
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, QuantileTransformer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error
from sklearn.impute import SimpleImputer
from sklearn.utils import resample
from xgboost import XGBRegressor
import matplotlib.pyplot as plt
import seaborn as sns

# STEP 3: Define Paths

In [ ]:
# 🔧 Update these paths!
data_root = "/content/drive/MyDrive/Data Science Capstone/Project 2/widsdatathon2025-university (1).zip (Unzipped Files)"
train_dir = os.path.join(data_root, "/content/drive/MyDrive/Data Science Capstone/Project 2/widsdatathon2025-university (1).zip (Unzipped Files)/train_tsv/train_tsv")
test_dir = os.path.join(data_root, "/content/drive/MyDrive/Data Science Capstone/Project 2/widsdatathon2025-university (1).zip (Unzipped Files)/test_tsv/test_tsv")
metadata_dir = os.path.join(data_root, "/content/drive/MyDrive/Data Science Capstone/Project 2/widsdatathon2025-university (1).zip (Unzipped Files)/metadata")
metadata_path = os.path.join(metadata_dir, "/content/drive/MyDrive/Data Science Capstone/Project 2/widsdatathon2025-university (1).zip (Unzipped Files)/metadata/training_metadata.csv")
test_metadata_path = os.path.join(metadata_dir, "/content/drive/MyDrive/Data Science Capstone/Project 2/widsdatathon2025-university (1).zip (Unzipped Files)/metadata/test_metadata.csv")
synthetic_data_path = os.path.join(data_root, "/content/drive/MyDrive/Data Science Capstone/Project 2/widsdatathon2025-university (1).zip (Unzipped Files)/Generated CSV")  # ⬅️ Your synthetic folder


# STEP 4: Function to Extract Upper Triangular

In [ ]:
def extract_upper_triangle(file_path):
    matrix = pd.read_csv(file_path, sep='\t', header=None).values
    upper_tri_indices = np.triu_indices_from(matrix, k=1)
    return matrix[upper_tri_indices]

# STEP 5: Load Real fMRI Training Data

In [ ]:
train_vectors = []
train_ids = []

print("🔄 Processing real fMRI training files...")
for filename in tqdm(os.listdir(train_dir)):
    if filename.endswith(".tsv"):
        participant_id = filename.split("_")[0].replace("sub-", "")
        file_path = os.path.join(train_dir, filename)
        vector = extract_upper_triangle(file_path)
        train_vectors.append(vector)
        train_ids.append(participant_id)

X_real = pd.DataFrame(train_vectors)
X_real["participant_id"] = train_ids

# STEP 6: Merge Metadata

In [ ]:
metadata = pd.read_csv(metadata_path)
metadata["participant_id"] = metadata["participant_id"].str.upper().str.strip()
X_real["participant_id"] = X_real["participant_id"].str.upper().str.strip()
train_df = pd.merge(X_real, metadata, on="participant_id")

# STEP 7: Load Synthetic Data

In [ ]:
synthetic_df_list = []
meta_cols = ['sex', 'age', 'site_id']
chunksize = 1000
print("🔄 Streaming synthetic data in chunks...")

# Track ages for distribution check
synthetic_ages = []

for filename in tqdm(os.listdir(synthetic_data_path)):
    if filename.endswith(".csv"):
        match = re.search(r"(\d+)p(\d+)_+(\d+)p(\d+)", filename)
        if not match:
            continue

        start_age = int(match.group(1)) + int(match.group(2)) / 100
        end_age = int(match.group(3)) + int(match.group(4)) / 100
        midpoint_age = (start_age + end_age) / 2

        # Record age for distribution
        chunk_iter = pd.read_csv(os.path.join(synthetic_data_path, filename), header=None, chunksize=chunksize, low_memory=True)
        for i, chunk in enumerate(chunk_iter):
            print(f"Processing chunk {i+1} of {filename} (Memory: {psutil.virtual_memory().percent}% used)")
            chunk["age"] = midpoint_age
            chunk["participant_id"] = [f"SYNTH_{filename}_{i*chunksize + j}" for j in range(len(chunk))]

            for col in meta_cols:
                if col not in chunk.columns:
                    chunk[col] = 0

            synthetic_df_list.append(chunk)
            synthetic_ages.extend([midpoint_age] * len(chunk))

            if len(synthetic_df_list) >= 5:
                partial_df = pd.concat(synthetic_df_list, ignore_index=True)
                train_df = pd.concat([train_df, partial_df], ignore_index=True)
                train_df = shuffle(train_df).reset_index(drop=True)
                synthetic_df_list = []
                del partial_df
                print(f"Concatenated batch (Memory: {psutil.virtual_memory().percent}% used)")

if synthetic_df_list:
    partial_df = pd.concat(synthetic_df_list, ignore_index=True)
    train_df = pd.concat([train_df, partial_df], ignore_index=True)
    train_df = shuffle(train_df).reset_index(drop=True)
    del partial_df, synthetic_df_list

# Check synthetic age distribution
synthetic_age_counts = pd.Series(synthetic_ages).value_counts().sort_index()
print("Synthetic data age distribution:\n", synthetic_age_counts)

# Warn if ages are missing
expected_ages = set(range(5, 22))  # Ages 5 to 21
present_ages = set(synthetic_age_counts.index)
missing_ages = expected_ages - present_ages
if missing_ages:
    print(f"Warning: Missing ages in synthetic data: {missing_ages}")
    print("Consider generating synthetic data for these ages to improve prediction diversity.")

print("✅ Combined train shape (real + synthetic):", train_df.shape)
print("Age distribution in train_df:\n", train_df['age'].value_counts())

# STEP 8: Combine Real + Synthetic Data

In [ ]:
# Step 8: Use train_df directly (no duplicate concatenation)
train_df_combined = train_df

In [ ]:
print(train_df_combined.columns)
print(train_df_combined.head())

# STEP 9: Balance Age Groups

In [ ]:
from sklearn.utils import resample

train_df_filtered = train_df_combined.copy()

# Ensure all ages 5 to 21 are represented
max_count = 200
resampled_list = []
present_ages = set(train_df_filtered["age"].astype(int))
missing_ages = set(range(5, 22)) - present_ages

# For missing ages, use real data (if available) or synthetic data as a fallback
for age in missing_ages:
    # Try to find similar ages in real data
    real_subset = train_df[train_df["age"].astype(int).between(age-2, age+2) & (train_df["participant_id"].str.startswith("sub-"))]
    if not real_subset.empty:
        resampled = resample(real_subset, replace=True, n_samples=max_count, random_state=42)
        resampled["age"] = age  # Assign the target age
        resampled_list.append(resampled)
    else:
        # Fallback to synthetic data or skip
        print(f"Warning: Age {age} missing and no similar real data found. Consider generating synthetic data for this age.")

# Resample existing ages
for age, group in train_df_filtered.groupby("age"):
    n_samples = min(max_count, len(group))
    resampled = resample(group, replace=True, n_samples=n_samples, random_state=42)
    resampled_list.append(resampled)

df_resampled = pd.concat(resampled_list).reset_index(drop=True)

print("✅ Resampled shape:", df_resampled.shape)
print("Age distribution in df_resampled:\n", df_resampled["age"].value_counts())

sns.histplot(df_resampled["age"], bins=20, kde=True)
plt.title("Training Age Distribution")
plt.xlabel("Age")
plt.ylabel("Frequency")
plt.grid(True)
plt.show()

# STEP 10: Preprocess with PCA + One-Hot Metadata

In [ ]:
# Coerce numeric columns to float
for col in df_resampled.columns:
    if isinstance(col, (int, str)) and (isinstance(col, int) or col.isdigit()):
        df_resampled[col] = pd.to_numeric(df_resampled[col], errors='coerce')

# Identify brain feature columns
brain_cols = [col for col in df_resampled.columns if (isinstance(col, int) or (isinstance(col, str) and col.isdigit())) and df_resampled[col].dtype in [np.float64, np.float32, np.int64]]

if not brain_cols:
    raise ValueError("No valid brain feature columns found. Check column types or naming.")

print(f"Found {len(brain_cols)} brain feature columns.")

# Define meta columns
meta_cols = [col for col in df_resampled.columns if col not in brain_cols + ["participant_id", "age"]]

# Scale brain features
scaler = StandardScaler()
brain_scaled = scaler.fit_transform(df_resampled[brain_cols])

# Impute missing values
imputer = SimpleImputer(strategy='mean')
brain_imputed = imputer.fit_transform(brain_scaled)

# Apply PCA
pca = PCA(n_components=100)
brain_pca = pca.fit_transform(brain_imputed)
print("PCA Explained Variance Ratio:", pca.explained_variance_ratio_.sum())

# Encode metadata
meta_encoded = pd.get_dummies(df_resampled[meta_cols])
meta_encoded = meta_encoded.reindex(columns=meta_encoded.columns, fill_value=0)

# Combine features
X = np.concatenate([brain_pca, meta_encoded.values], axis=1)
y = df_resampled["age"]

# STEP 11: Transform Target + Impute

In [ ]:
# Skip QuantileTransformer to preserve raw age distribution
y_transformed = y  # Use raw ages

imputer = SimpleImputer(strategy='mean')
X = imputer.fit_transform(X)

# STEP 12: Train Ridge, RF, XGBoost

In [ ]:
# Train Ridge
ridge = Ridge(alpha=1.0)
ridge.fit(X, y_transformed)  # Use transformed target

# Train XGBoost
xgb = XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42)
xgb.fit(X, y_transformed)

# Evaluate with cross-validation
scores = cross_val_score(xgb, X, y_transformed, cv=5, scoring="neg_mean_squared_error")
print("Cross-Validation RMSE:", np.sqrt(-scores.mean()))

# STEP 13: Process Test Set

In [ ]:
# Load test brain data
test_vectors = []
test_ids = []

for filename in tqdm(os.listdir(test_dir)):
    if filename.endswith(".tsv"):
        participant_id = filename.split("_")[0].replace("sub-", "")
        file_path = os.path.join(test_dir, filename)
        vector = extract_upper_triangle(file_path)
        test_vectors.append(vector)
        test_ids.append(participant_id)

X_test_raw = pd.DataFrame(test_vectors)
X_test_raw["participant_id"] = test_ids

# Load and merge metadata
test_metadata = pd.read_csv(test_metadata_path)
X_test_raw["participant_id"] = X_test_raw["participant_id"].str.upper().str.strip()
test_metadata["participant_id"] = test_metadata["participant_id"].str.upper().str.strip()
test_df = pd.merge(X_test_raw, test_metadata, on="participant_id")

# STEP 14: Predict on Test Set

In [ ]:
test_df.columns = test_df.columns.astype(str)
test_brain = test_df.reindex(columns=brain_cols, fill_value=0)

# Check for NaNs after reindexing
print("NaNs in test_brain:", test_brain.isna().sum().sum())

test_brain_scaled = scaler.transform(test_brain)

# Check for NaNs after scaling
print("NaNs in test_brain_scaled:", np.isnan(test_brain_scaled).sum())

test_brain_pca = pca.transform(test_brain_scaled)

# Check for NaNs after PCA
print("NaNs in test_brain_pca:", np.isnan(test_brain_pca).sum())

for col in meta_cols:
    if col not in test_df.columns:
        test_df[col] = 0

test_meta_encoded = pd.get_dummies(test_df[meta_cols])
test_meta_encoded = test_meta_encoded.reindex(columns=meta_encoded.columns, fill_value=0)

# Check for NaNs in metadata
print("NaNs in test_meta_encoded:", test_meta_encoded.isna().sum().sum())

X_test_final = np.concatenate([test_brain_pca, test_meta_encoded.values], axis=1)

# Impute NaNs in X_test_final
imputer = SimpleImputer(strategy='mean')
X_test_final = imputer.fit_transform(X_test_final)

# Check for NaNs after imputation
print("NaNs in X_test_final after imputation:", np.isnan(X_test_final).sum())

# Make ensemble predictions
ridge_predictions = ridge.predict(X_test_final)
xgb_predictions = xgb.predict(X_test_final)
raw_predictions = 0.5 * ridge_predictions + 0.5 * xgb_predictions

# No inverse transform since QuantileTransformer is removed
predictions = np.clip(raw_predictions, 5, 25)

print("Raw Predictions:", raw_predictions[:10])
print("Predicted Ages:", predictions[:10])
print("Predicted age distribution:\n", pd.Series(predictions).value_counts(bins=range(5, 26, 1)).sort_index())

# STEP 15: Export Submission

In [ ]:
submission = pd.DataFrame({
    "participant_id": test_ids,
    "age": predictions
})
submission_path = "/content/drive/MyDrive/submission_with_synthetic.csv"
submission.to_csv(submission_path, index=False)
print(f"✅ Submission file saved to: {submission_path}")

# STEP 16: Visualize Predictions

In [ ]:
sns.histplot(predictions, bins=20, kde=True)
plt.title("Predicted Age Distribution")
plt.xlabel("Age")
plt.ylabel("Frequency")
plt.grid(True)
plt.show()

In [ ]:
print("Unique ages in training:", train_df['age'].unique())
print("Mean/Min/Max of y_train:", y.min(), y.max())
print("Sample predictions:", predictions[:10])